# DKTC 모델 학습 - 한국어 위협 대화 분류
## KcELECTRA-base Baseline

In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 시드 고정
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

set_seed(42)

# 디바이스 설정
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print('Using MPS (Apple Silicon)')
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print('Using CUDA')
else:
    device = torch.device('cpu')
    print('Using CPU')

print(f'PyTorch version: {torch.__version__}')

## 1. 데이터 로드 및 전처리

In [ ]:
DATA_DIR = 'aiffel-d-lthon-dktc-online-17'

# 원본 데이터
train_orig = pd.read_csv(f'{DATA_DIR}/train.csv')
test_df = pd.read_csv(f'{DATA_DIR}/test.csv')
submission = pd.read_csv(f'{DATA_DIR}/submission.csv')

# 합성 일반 대화 데이터
normal_df = pd.read_csv('synthetic_normal_conversations.csv')

print(f'원본 Train: {train_orig.shape}')
print(f'합성 일반 대화: {normal_df.shape}')
print(f'Test: {test_df.shape}')

# 중복 제거
train_dedup = train_orig.drop_duplicates(subset='conversation').reset_index(drop=True)
print(f'중복 제거 후 Train: {train_dedup.shape}')

# 합성 데이터 병합
train_full = pd.concat([train_dedup[['class', 'conversation']], 
                        normal_df[['class', 'conversation']]], ignore_index=True)
print(f'\n최종 Train 데이터: {train_full.shape}')
print(train_full['class'].value_counts())

In [ ]:
# 전처리 함수 - Train/Test 포맷 통일
def preprocess_text(text):
    """줄바꿈을 공백으로 변환하여 Train/Test 포맷 통일"""
    text = str(text)
    text = text.replace('\n', ' ')  # newline → space
    text = re.sub(r'\s+', ' ', text).strip()  # 다중 공백 제거
    return text

# 전처리 적용
train_full['text'] = train_full['conversation'].apply(preprocess_text)
test_df['text'] = test_df['conversation'].apply(preprocess_text)

# 레이블 인코딩
label2id = {
    '갈취 대화': 0,
    '기타 괴롭힘 대화': 1,
    '일반 대화': 2,
    '직장 내 괴롭힘 대화': 3,
    '협박 대화': 4
}
id2label = {v: k for k, v in label2id.items()}
train_full['label'] = train_full['class'].map(label2id)

print('레이블 매핑:')
for name, idx in label2id.items():
    print(f'  {idx}: {name}')

print(f'\n전처리 예시:')
print(f'Before: {train_full["conversation"].iloc[0][:100]}...')
print(f'After:  {train_full["text"].iloc[0][:100]}...')

## 2. 토크나이저 및 데이터셋

In [ ]:
MODEL_NAME = 'beomi/KcELECTRA-base'
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 토큰 길이 분포 확인
sample_lengths = [len(tokenizer.encode(text)) for text in train_full['text'].sample(500, random_state=42)]
print(f'토큰 길이 통계: mean={np.mean(sample_lengths):.0f}, '
      f'median={np.median(sample_lengths):.0f}, '
      f'95th={np.percentile(sample_lengths, 95):.0f}, '
      f'max={np.max(sample_lengths)}')
print(f'max_length={MAX_LENGTH}에서 커버율: {sum(1 for l in sample_lengths if l <= MAX_LENGTH)/len(sample_lengths)*100:.1f}%')

In [ ]:
class DKTCDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Train/Val 분할
X_train, X_val, y_train, y_val = train_test_split(
    train_full['text'].tolist(),
    train_full['label'].tolist(),
    test_size=0.15,
    random_state=42,
    stratify=train_full['label'].tolist()
)

train_dataset = DKTCDataset(X_train, y_train, tokenizer, MAX_LENGTH)
val_dataset = DKTCDataset(X_val, y_val, tokenizer, MAX_LENGTH)

print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}')
print(f'Train 레이블 분포: {pd.Series(y_train).value_counts().sort_index().to_dict()}')
print(f'Val 레이블 분포: {pd.Series(y_val).value_counts().sort_index().to_dict()}')

## 3. 모델 학습

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1_macro = f1_score(labels, preds, average='macro')
    f1_weighted = f1_score(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)
    return {
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'accuracy': acc
    }

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=5,
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir='./outputs/kcelectra-baseline',
    num_train_epochs=10,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    logging_steps=50,
    seed=42,
    report_to='none',
    fp16=False,  # MPS에서는 fp16 미지원
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print(f'Model: {MODEL_NAME}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Device: {training_args.device}')

In [ ]:
# 학습 시작
train_result = trainer.train()

print(f'\n=== 학습 완료 ===')
print(f'Total steps: {train_result.global_step}')
print(f'Training loss: {train_result.training_loss:.4f}')

## 4. 평가

In [ ]:
# Validation 평가
eval_results = trainer.evaluate()
print('=== Validation 결과 ===')
for key, val in eval_results.items():
    print(f'  {key}: {val:.4f}')

# 상세 Classification Report
val_preds = trainer.predict(val_dataset)
y_pred = np.argmax(val_preds.predictions, axis=-1)

print(f'\n=== Classification Report ===')
target_names = [id2label[i] for i in range(5)]
print(classification_report(y_val, y_pred, target_names=target_names, digits=4))

In [ ]:
# Confusion Matrix 시각화
cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names)
plt.title('Confusion Matrix (Validation)', fontsize=14, fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('figures/06_confusion_matrix_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. 학습 곡선

In [ ]:
# 학습 로그에서 메트릭 추출
log_history = trainer.state.log_history

train_losses = [(l['step'], l['loss']) for l in log_history if 'loss' in l and 'eval_loss' not in l]
eval_metrics = [(l['epoch'], l.get('eval_loss'), l.get('eval_f1_macro')) 
                for l in log_history if 'eval_loss' in l]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training Loss
if train_losses:
    steps, losses = zip(*train_losses)
    axes[0].plot(steps, losses, 'b-', alpha=0.7)
    axes[0].set_title('Training Loss', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')

# Eval F1
if eval_metrics:
    epochs, eval_losses, f1_scores = zip(*eval_metrics)
    ax2 = axes[1]
    ax2.plot(epochs, f1_scores, 'g-o', label='F1 Macro', linewidth=2)
    ax2.set_title('Validation F1 Score', fontsize=13, fontweight='bold')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('F1 Macro')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figures/07_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Test 예측 및 Submission 생성

In [ ]:
# Test 데이터 예측
test_texts = test_df['text'].tolist()
test_dataset = DKTCDataset(
    test_texts, 
    [0] * len(test_texts),  # dummy labels
    tokenizer, 
    MAX_LENGTH
)

test_preds = trainer.predict(test_dataset)
test_pred_labels = np.argmax(test_preds.predictions, axis=-1)
test_pred_classes = [id2label[p] for p in test_pred_labels]

# 예측 분포 확인
pred_dist = pd.Series(test_pred_classes).value_counts()
print('Test 예측 분포:')
print(pred_dist)

# Submission 생성
submission['class'] = test_pred_classes
submission_path = 'outputs/submission_baseline.csv'
os.makedirs('outputs', exist_ok=True)
submission.to_csv(submission_path, index=False)
print(f'\nSubmission 저장: {submission_path}')
submission.head(10)

## 7. Ablation Study 기록

In [ ]:
# Ablation Study 결과 기록
experiment = {
    'exp_id': 'B01',
    'model': MODEL_NAME,
    'normal_count': len(normal_df),
    'augmentation': 'None',
    'lr': 2e-5,
    'epochs': training_args.num_train_epochs,
    'max_length': MAX_LENGTH,
    'val_f1_macro': eval_results.get('eval_f1_macro', 0),
    'val_accuracy': eval_results.get('eval_accuracy', 0),
    'notes': 'Baseline - KcELECTRA + synthetic normal'
}

# 결과를 CSV로 저장
ablation_path = 'outputs/ablation_study.csv'
if os.path.exists(ablation_path):
    ablation_df = pd.read_csv(ablation_path)
    ablation_df = pd.concat([ablation_df, pd.DataFrame([experiment])], ignore_index=True)
else:
    ablation_df = pd.DataFrame([experiment])

ablation_df.to_csv(ablation_path, index=False)
print('=== Ablation Study ===' )
ablation_df